In [2]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd

from tqdm import tqdm
from pathlib import Path
from datetime import timedelta
from shapely import wkt
import ast
from shapely.geometry import Point, LineString

datasets = Path("/nas/cee-water/cjgleason/data")
era5_dir = datasets / "ERA5-Land/sub_basin_timeseries"
swot_lake_dir = datasets / 'hydrocron' / 'lake'

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/reservoirs")
preprocess_dir = save_dir / "preprocess"
metadata_dir = save_dir / "metadata"

matchups = gpd.read_file(metadata_dir / "All_MERIT_matchups.gpkg").set_index('comid')
matchups.index = matchups.index.astype(str)
 
# Safely convert stringified lists/dicts back to Python objects
def safe_literal_eval(df, col):
    df[col] = df[col].apply(lambda x: ast.literal_eval(x))
    return df
    
for col in ["mb_values", "lake_reach_ids", "lake_pld_ids"]:
    matchups = safe_literal_eval(matchups, col)

In [8]:
matchups['mb_values'].apply(len).gt(0).sum()

np.int64(897)

In [ ]:
glow_dir = datasets / "GLOW-S" / "daily_reach_aggregated"

glow_dfs = []
for region_file in glow_dir.glob('*_daily_median.parquet'):
    glow_dfs.append(pd.read_parquet(region_file))
                    
glow = pd.concat(glow_dfs)
glow

In [ ]:
from data import BasinDeltaTable

root_dir = save_dir / 'deltalakes' / 'training'
store = BasinDeltaTable(root_dir)